# Etapa E10, de la clasificación a la acción — modelo neonatal NCHS (M4)

El modelo quedó clasificado como **predictor** en la puerta P4 (framework 1.3). La etapa E10 traduce esa clase en un uso concreto y verifica lo que el framework exige antes de recomendarlo (README, E10): la **carga de trabajo** (cuántas alertas por cada muerte encontrada) y el **desempeño por subgrupos** (si el modelo funciona igual en todos los grupos, para no reproducir inequidades). Además describe **qué impulsa el riesgo**, para orientar la estrategia de prevención. Versión en inglés: `06_e10_action.ipynb` (mismo código). Requiere el notebook 04 (`data/outputs_04_selection.json`).

- **Modelo:** el algoritmo principal (XGBoost, hiperparámetros del notebook 04), ajustado una sola vez con 2016–2021 y recalibrado dentro de ese período, aplicado una sola vez a la **prueba temporal 2022–2023**: los nacimientos que el modelo encontraría en la práctica.
- **Carga de trabajo:** puntos de operación (1%, 2%, 5% y 10% de mayor riesgo y cortes de riesgo de 1%, 2% y 5%) para todos los nacidos vivos y para los recién nacidos vivos al comenzar el día 2 (punto de referencia a las 48 horas), donde el cuidado preventivo todavía tiene tiempo de actuar.
- **Subgrupos:** raza y origen hispano de la madre, forma de pago del parto, edad gestacional, pluralidad y cohorte. Alertas de reporte fijadas antes de correr, para todos los grupos salvo la edad gestacional: AUROC más de 0,05 por debajo de la población completa, u observados/esperados fuera de 0,80–1,25. Las bandas de edad gestacional son solo descriptivas, porque la edad gestacional es uno de los predictores más fuertes y el AUROC dentro de una banda es menor por construcción.
- **Qué impulsa el riesgo:** contribución SHAP absoluta media de cada predictor (descriptiva, no causal), marcando los factores relacionados con la atención prenatal o del parto, donde una entidad de salud puede actuar.

Salida: `data/outputs_06_e10_action.txt`. Unos 20 a 40 minutos.

In [1]:
import sys, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.optimize import brentq
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "neonatal_nchs_v1"))
OUT_FILE = ROOT / "data" / "outputs_06_e10_action.txt"
SELECTION_FILE = ROOT / "data" / "outputs_04_selection.json"
SEED = 20261123
DEV_YEARS, TEST_YEARS = list(range(2016, 2022)), [2022, 2023]
FRACTION = float(os.environ.get("NCHS_FRACTION", 0.02))
QUICK = os.environ.get("NCHS_QUICK") == "1"
N_JOBS = int(os.environ.get("NCHS_JOBS", -1))
B = 30 if QUICK else 200
META = ["unit_id", "group_id", "period", "label_main", "at_risk_48h", "label_day2_27", "label_early_neonatal",
        "ge28", "ga_imputed", "bw_imputed", "n_unknown"]
PREDICTORS = [c for c in pq.read_schema(CONTRACT_DIR / "cohort_2021.parquet").names if c not in META]
REFERENCE = ["gestational_age", "birthweight"]           # reference model fixed in the outcome card
EXTRA = ["period", "label_main", "at_risk_48h", "label_day2_27", "label_early_neonatal", "ge28", "n_unknown"]

# Primary learner and hyperparameters chosen by the preliminary run (notebook 04, selection rule)
sel = json.loads(SELECTION_FILE.read_text(encoding="utf-8"))
assert QUICK or not sel["quick"], "notebook 04 was run in QUICK mode: run it again with the full sample"
primary, BEST = sel["primary"], sel["best"]

def make_model(name):
    cfg = dict(BEST[name]); n_iter = cfg.pop("n_estimators", None)
    imp = SimpleImputer(strategy="median")                       # decision M4-D3: no missingness indicators
    if name == "LR":
        return make_pipeline(imp, StandardScaler(), LogisticRegression(solver="liblinear", max_iter=500, **cfg))
    n_trees = 50 if QUICK else 300
    if name == "RF":
        return make_pipeline(imp, RandomForestClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    if name == "ET":
        return make_pipeline(imp, ExtraTreesClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    return make_pipeline(imp, XGBClassifier(n_estimators=(min(n_iter, 50) if QUICK else n_iter), tree_method="hist",
                                            n_jobs=N_JOBS, random_state=SEED, **cfg))

pd.set_option("display.max_colwidth", None)
lines = []
def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)
show("Primary learner (notebook 04)", pd.Series({"primary learner": primary, "hyperparameters": json.dumps(BEST[primary]),
                                                  "ensemble adopted in notebook 04": sel["ensemble_adopted"]}))


== Primary learner (notebook 04)
primary learner                                                                                                                                                                                                       XGB
hyperparameters                    {"max_depth": 8, "learning_rate": 0.0211, "min_child_weight": 4.5508, "subsample": 0.8075, "colsample_bytree": 0.6844, "reg_lambda": 0.1686, "reg_alpha": 0.0074, "n_estimators": 302}
ensemble adopted in notebook 04                                                                                                                                                                                     False


## 1. Muestras

In [2]:
# Samples: every event and FRACTION of non-events, weight 1/FRACTION for non-events (the weighted
# sample stands for the full population). Development 2016-2021; temporal test 2022-2023.
def load(years, seed):
    rng = np.random.default_rng(seed); parts = []; n = e = 0
    for y in years:     # read in batches and keep the sample only, so the memory stays low
        for b in pq.ParquetFile(CONTRACT_DIR / f"cohort_{y}.parquet").iter_batches(500_000, columns=PREDICTORS + EXTRA):
            c = b.to_pandas()
            n += len(c); e += int(c["label_main"].sum())
            parts.append(c[c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)])
    d = pd.concat(parts, ignore_index=True)
    if QUICK:
        d = d.sample(n=min(len(d), 40000), random_state=seed).reset_index(drop=True)
    d["w"] = np.where(d["label_main"], 1.0, 1.0 / FRACTION)
    return d, n, e

dev, n_dev, e_dev = load(DEV_YEARS, SEED)
test, n_test, e_test = load(TEST_YEARS, SEED + 1)
show("Data", pd.DataFrame({"births": [n_dev, n_test], "neonatal deaths": [e_dev, e_test],
                           "sample rows": [len(dev), len(test)]}, index=["development 2016-2021", "temporal test 2022-2023"]))


== Data
                           births  neonatal deaths  sample rows
development 2016-2021    22578517            62713       512880
temporal test 2022-2023   7251350            19846       164237


## 2. Modelo aplicado a la prueba temporal

In [3]:
# Model of stage S10: the primary learner fitted once on the whole development sample (2016-2021) and
# recalibrated inside it, applied once to the temporal test (2022-2023), as in notebook 05 (criterion C4).
# The test cohorts play the role of the births the model would meet in practice.
# Metrics (weighted where the sampling matters; AUROC does not depend on it)
logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
expit = lambda z: 1 / (1 + np.exp(-z))

def cal_slope_intercept(y, p, w):
    lp = logit(p)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), y, sample_weight=w).coef_[0, 0]
    f = lambda a: np.sum(w * (y - expit(a + lp)))        # calibration-in-the-large (offset = lp)
    return float(slope), float(brentq(f, -10, 10))

def metrics(y, p, w):
    o = np.argsort(p); cw = np.cumsum(w[o]); m = {}
    for share in (0.05, 0.10):
        cut = p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
        m[f"events in top {int(share * 100)}% (%)"] = 100 * float(w[(p >= cut) & y].sum() / w[y].sum())
    slope, intercept = cal_slope_intercept(y, p, w)
    g = np.empty(len(p), int); g[o] = np.minimum((10 * cw / cw[-1]).astype(int), 9)   # weighted deciles
    ece = sum(abs(np.sum(w[g == k] * y[g == k]) - np.sum(w[g == k] * p[g == k])) for k in range(10)) / w.sum()
    m.update({"AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p, sample_weight=w),
              "prevalence": float(np.sum(w * y) / w.sum()), "Brier": float(np.sum(w * (p - y) ** 2) / w.sum()),
              "calibration slope": slope, "calibration intercept": intercept,
              "observed/expected": float(np.sum(w * y) / np.sum(w * p)), "ECE": float(ece)})
    return m

def boot_ci(y, p, w, keys, seed):
    rng = np.random.default_rng(seed); pos, neg = np.where(y)[0], np.where(~y)[0]; res = []
    for _ in range(B):
        ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        res.append({k: v for k, v in metrics(y[ix], p[ix], w[ix]).items() if k in keys})
    r = pd.DataFrame(res)
    return r.quantile(0.025), r.quantile(0.975)

def fit_calibrated(name, X, y, w, seed, cols=None):
    """Fit on 80% of the training part and Platt-recalibrate on the other 20% (inside training)."""
    cols = cols or PREDICTORS
    X = X[cols]
    X_f, X_c, y_f, y_c, w_f, w_c = train_test_split(X, y, w, test_size=0.2, stratify=y, random_state=seed)
    m = make_model(name); step = m.steps[-1][0]
    m.fit(X_f, y_f, **{f"{step}__sample_weight": w_f})
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(m.predict_proba(X_c)[:, 1]).reshape(-1, 1), y_c, sample_weight=w_c)
    return lambda Xn: platt.predict_proba(logit(m.predict_proba(Xn[cols])[:, 1]).reshape(-1, 1))[:, 1]


f = fit_calibrated(primary, dev[PREDICTORS], dev["label_main"].to_numpy(bool), dev["w"].to_numpy(), SEED)
p_t = f(test[PREDICTORS]); y_t = test["label_main"].to_numpy(bool); w_t = test["w"].to_numpy()
lm_t = test["at_risk_48h"].to_numpy(bool); y48 = test["label_day2_27"].to_numpy(bool)
show("Temporal test 2022-2023 (check: same model as the temporal validation of notebook 05)",
     pd.Series({"AUROC": round(roc_auc_score(y_t, p_t), 4), "births (weighted)": int(round(w_t.sum())),
                "neonatal deaths": int(y_t.sum())}))


== Temporal test 2022-2023 (check: same model as the temporal validation of notebook 05)
AUROC                9.606000e-01
births (weighted)    7.239396e+06
neonatal deaths      1.984600e+04


## 3. Carga de trabajo en varios puntos de operación

In [4]:
# Workload (framework, S10): for a few operating points, how many births are flagged, how many deaths are found
# and how many alerts are needed per death found (1 / PPV). Two populations: all live births at birth + 5 minutes,
# and the newborns alive at the start of day 2 (48-hour landmark), where preventive care has time to act.
def operating_points(y, p, w, label):
    rows = []; n = w.sum(); deaths = w[y].sum()
    o = np.argsort(-p); cw = np.cumsum(w[o])
    points = [("top 1%", None, 0.01), ("top 2%", None, 0.02), ("top 5%", None, 0.05), ("top 10%", None, 0.10),
              ("risk >= 1%", 0.01, None), ("risk >= 2%", 0.02, None), ("risk >= 5%", 0.05, None)]
    for name, thr, share in points:
        cut = thr if thr is not None else p[o][np.searchsorted(cw, share * n)]
        flag = p >= cut
        tp, fp = w[flag & y].sum(), w[flag & ~y].sum(); fn, tn = w[~flag & y].sum(), w[~flag & ~y].sum()
        rows.append({"population": label, "operating point": name, "risk cut": round(float(cut), 4),
                     "% flagged": round(100 * (tp + fp) / n, 2), "flagged per 1,000 births": round(1000 * (tp + fp) / n, 1),
                     "sensitivity": round(tp / deaths, 3), "specificity": round(tn / (tn + fp), 4),
                     "PPV": round(tp / (tp + fp), 4), "alerts per death found": round((tp + fp) / tp, 1) if tp else None,
                     "deaths found per 1,000 births": round(1000 * tp / n, 2),
                     "net benefit": round(tp / n - fp / n * cut / (1 - cut), 5)})
    return rows
wk = operating_points(y_t, p_t, w_t, "all live births (birth + 5 min)") + \
     operating_points(y48[lm_t], p_t[lm_t], w_t[lm_t], "alive at the start of day 2 (deaths on days 2-27)")
show("Workload at several operating points (temporal test 2022-2023, weighted to the full population)",
     pd.DataFrame(wk).set_index(["population", "operating point"]))


== Workload at several operating points (temporal test 2022-2023, weighted to the full population)
                                                                   risk cut  % flagged  flagged per 1,000 births  sensitivity  specificity     PPV  alerts per death found  deaths found per 1,000 births  net benefit
population                                        operating point                                                                                                                                                     
all live births (birth + 5 min)                   top 1%             0.0297       1.00                      10.0        0.747       0.9920  0.2046                     4.9                           2.05      0.00180
                                                  top 2%             0.0091       2.00                      20.0        0.822       0.9822  0.1126                     8.9                           2.25      0.00209
                                        

## 4. Desempeño por subgrupos

In [5]:
# Subgroups (framework, S10): does the model work as well across groups? Discrimination, calibration (observed /
# expected, calibration slope) and sensitivity at the overall top-10% cut, on the temporal test. Reporting flags
# fixed before running, for the equity groups (race and Hispanic origin, payment) and plurality and cohort: AUROC more
# than 0.05 below the whole population, or observed/expected outside 0.80-1.25. Gestational-age bands are descriptive
# only: gestational age is one of the strongest predictors, so the AUROC within a band is lower by construction.
o = np.argsort(-p_t); cw = np.cumsum(w_t[o]); cut10 = p_t[o][np.searchsorted(cw, 0.10 * w_t.sum())]
T = test
race = np.select([T["race_black"] == 1, T["race_hispanic"] == 1, T["race_asian"] == 1, T["race_other"] == 1,
                  T[["race_black", "race_hispanic", "race_asian", "race_other"]].isna().any(axis=1)],
                 ["Black", "Hispanic", "Asian", "Other", "unknown"], "White")
pay = np.select([T["pay_medicaid"] == 1, T["pay_self"] == 1, T["pay_other"] == 1,
                 T[["pay_medicaid", "pay_self", "pay_other"]].isna().any(axis=1)],
                ["Medicaid", "self-pay", "other", "unknown"], "private insurance")
ga = T["gestational_age"].to_numpy()
ga_band = np.select([ga < 28, ga < 32, ga < 37], ["22-27 weeks", "28-31 weeks", "32-36 weeks"], "37 weeks or more")
plural = np.where(T["triplet_or_more"] == 1, "triplet or more", np.where(T["twin"] == 1, "twin", "singleton"))
groups = {"mother's race and Hispanic origin": race, "payment of the delivery": pay, "gestational age": ga_band,
          "plurality": plural, "cohort": T["period"].astype(str).to_numpy()}
base_auc = roc_auc_score(y_t, p_t)
rows = []
for gname, g in groups.items():
    for level in sorted(pd.unique(g)):
        m = g == level; yy, pp, ww = y_t[m], p_t[m], w_t[m]
        if yy.sum() < 10 or (~yy).sum() < 10:
            rows.append({"group": gname, "level": level, "births (weighted)": int(round(ww.sum())), "deaths": "<10"}); continue
        auc = roc_auc_score(yy, pp); oe = float(np.sum(ww * yy) / np.sum(ww * pp))
        slope, _ = cal_slope_intercept(yy, pp, ww)
        flag = []
        if gname != "gestational age":
            if auc < base_auc - 0.05: flag.append("AUROC")
            if not 0.80 <= oe <= 1.25: flag.append("O/E")
        rows.append({"group": gname, "level": level, "births (weighted)": int(round(ww.sum())), "deaths": int(yy.sum()),
                     "deaths per 1,000": round(1000 * np.sum(ww * yy) / ww.sum(), 2), "AUROC": round(auc, 4),
                     "observed/expected": round(oe, 3), "calibration slope": round(slope, 3),
                     "sensitivity at top-10% cut": round(float(np.sum(ww * (yy & (pp >= cut10))) / np.sum(ww * yy)), 3),
                     "% flagged at top-10% cut": round(100 * float(np.sum(ww * (pp >= cut10)) / ww.sum()), 1),
                     "flag": ", ".join(flag) or "—"})
show(f"Performance by subgroup (temporal test; whole-population AUROC {base_auc:.4f}; flags: AUROC < "
     f"{base_auc - 0.05:.3f} or O/E outside 0.80-1.25; gestational age descriptive only)", pd.DataFrame(rows).set_index(["group", "level"]))


== Performance by subgroup (temporal test; whole-population AUROC 0.9606; flags: AUROC < 0.911 or O/E outside 0.80-1.25; gestational age descriptive only)
                                                     births (weighted)  deaths  deaths per 1,000   AUROC  observed/expected  calibration slope  sensitivity at top-10% cut  % flagged at top-10% cut flag
group                             level                                                                                                                                                                  
mother's race and Hispanic origin Asian                         432642     792              1.83  0.9650              0.956              1.063                       0.914                       7.6    —
                                  Black                         999690    4440              4.44  0.9536              0.970              0.971                       0.917                      16.1    —
                                  Hi

## 5. Qué impulsa el riesgo y salida

In [6]:
# Original objective of the thesis (sensitivity and specificity both above 0.75): the best pair any threshold can
# reach. Weighted to the full population; temporal test 2022-2023. Descriptive only.
def best_pairs(y, p, w):
    o = np.argsort(-p); yo, wo, po = y[o], w[o], p[o]
    tp = np.cumsum(wo * yo); fp = np.cumsum(wo * ~yo)
    sens = tp / tp[-1]; spec = 1 - fp / fp[-1]
    last = np.r_[po[1:] != po[:-1], True]          # evaluate only at the end of tied scores
    sens, spec, po = sens[last], spec[last], po[last]
    i = np.argmax(np.minimum(sens, spec))
    j = np.where(sens >= 0.75)[0][0]; k = np.where(spec >= 0.75)[0][-1]
    return {"balanced point: sensitivity": round(float(sens[i]), 3), "balanced point: specificity": round(float(spec[i]), 3),
            "balanced point: risk cut": round(float(po[i]), 4),
            "specificity when sensitivity >= 0.75": round(float(spec[j]), 3),
            "sensitivity when specificity >= 0.75": round(float(sens[k]), 3),
            "some threshold gives both >= 0.75": bool(((sens >= 0.75) & (spec >= 0.75)).any())}
show("Original objective: best sensitivity-specificity pairs (temporal test 2022-2023)", pd.DataFrame({
    "all live births": best_pairs(y_t, p_t, w_t), "alive at the start of day 2": best_pairs(y48[lm_t], p_t[lm_t], w_t[lm_t])}))

# What drives the risk (to orient the prevention strategy): mean absolute SHAP contribution of each predictor
# in the temporal test (XGBoost), on the log-odds scale. Descriptive, not causal: factors related to prenatal or
# delivery care (where a health entity can act) are marked.
MODIFIABLE = {"no_prenatal_care", "care_after_first_trimester", "prenatal_visits", "smoked_during_pregnancy", "bmi",
              "weight_gain", "antenatal_steroids", "born_out_of_hospital", "attendant_midwife", "attendant_other",
              "induction", "delivery_cesarean", "delivery_forceps", "delivery_vacuum", "wic", "syphilis", "gonorrhea",
              "chlamydia", "hepatitis_b", "hepatitis_c", "chorioamnionitis", "antibiotics_in_labor", "interval_lt18"}
if primary == "XGB":
    import xgboost as xgb
    model = make_model(primary); step = model.steps[-1][0]
    model.fit(dev[PREDICTORS], dev["label_main"].to_numpy(bool), **{f"{step}__sample_weight": dev["w"].to_numpy()})
    Xt = model[:-1].transform(test[PREDICTORS])
    contrib = model[-1].get_booster().predict(xgb.DMatrix(Xt), pred_contribs=True)[:, :-1]
    imp = pd.Series(np.average(np.abs(contrib), axis=0, weights=w_t), index=PREDICTORS).sort_values(ascending=False)
    tab = pd.DataFrame({"mean |SHAP| (log-odds)": imp.round(4),
                        "share of total (%)": (100 * imp / imp.sum()).round(1),
                        "related to prenatal or delivery care": ["yes" if v in MODIFIABLE else "" for v in imp.index]})
    show("Contribution of each predictor (top 25; temporal test, weighted)", tab.head(25))
    show("Share of the total contribution by type of predictor (%)", pd.Series({
        "gestational age and birthweight": round(float(tab.loc[REFERENCE, "share of total (%)"].sum()), 1),
        "related to prenatal or delivery care": round(float(tab.loc[tab.index.isin(MODIFIABLE), "share of total (%)"].sum()), 1),
        "other": round(float(100 - tab.loc[REFERENCE, "share of total (%)"].sum() - tab.loc[tab.index.isin(MODIFIABLE), "share of total (%)"].sum()), 1)}))

OUT_FILE.write_text("Stage S10 (from classification to action) of the neonatal model, temporal test 2022-2023\n"
                    + "\n".join(lines) + "\n", encoding="utf-8")
print("written:", OUT_FILE)


== Original objective: best sensitivity-specificity pairs (temporal test 2022-2023)
                                     all live births alive at the start of day 2
balanced point: sensitivity                    0.904                       0.856
balanced point: specificity                    0.904                       0.856
balanced point: risk cut                      0.0012                      0.0008
specificity when sensitivity >= 0.75           0.992                        0.97
sensitivity when specificity >= 0.75           0.946                         0.9
some threshold gives both >= 0.75               True                        True

== Contribution of each predictor (top 25; temporal test, weighted)
                            mean |SHAP| (log-odds)  share of total (%) related to prenatal or delivery care
apgar5                                      1.2679                40.5                                     
birthweight                                 0.7158             